# ORBIT x LeRobot Validation Notebook

This notebook validates **ORBIT**'s dataset profiling capabilities against well-known **LeRobot** datasets.

**What this notebook does:**
1. Downloads 5 LeRobot datasets of varying quality and size
2. Runs ORBIT's capability profiler on each dataset
3. Performs sim-to-real transfer analysis
4. Generates benchmark tables, radar charts, and heatmaps
5. Exports all results as CSV, PNG, and JSON

**Requirements:** Google Colab free tier (T4 GPU, ~15 GB RAM). Click **Runtime > Run all** to execute end-to-end.

---

## 1. Setup

Install dependencies and verify GPU availability.

In [ ]:
# Install dependencies
!pip install -q orbit-robotics[profile] lerobot torch torchvision

import os
import time
import json
import logging
import warnings
warnings.filterwarnings("ignore")

import torch
import numpy as np

# GPU info
print(f"GPU available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")
else:
    print("No GPU detected — running on CPU (will be slower)")

device = "cuda" if torch.cuda.is_available() else "cpu"

# Create output directory
os.makedirs("results", exist_ok=True)

# Configure logging
logging.basicConfig(level=logging.INFO, format="%(levelname)s | %(name)s | %(message)s")
logging.getLogger("orbit").setLevel(logging.INFO)

print(f"\nDevice: {device}")
print("Setup complete.")

## 2. Download LeRobot Datasets

We download 5 well-known LeRobot datasets spanning different robot platforms, tasks, and dataset sizes:

| Dataset | Robot | Task | Notes |
|---------|-------|------|-------|
| `lerobot/pusht` | 2D pusher | Push T-shaped block | Simple 2D environment |
| `lerobot/aloha_sim_transfer_cube_human` | ALOHA (sim) | Transfer cube | Bimanual sim demos |
| `lerobot/xarm_lift_medium_replay` | xArm | Lift object | Medium-quality replays |
| `lerobot/aloha_sim_insertion_human` | ALOHA (sim) | Peg insertion | Precision task |
| `lerobot/umi_cup_in_the_wild` | UMI gripper | Cup manipulation | Real-world, in-the-wild |

In [ ]:
from lerobot.common.datasets.lerobot_dataset import LeRobotDataset

datasets_to_test = [
    "lerobot/pusht",
    "lerobot/aloha_sim_transfer_cube_human",
    "lerobot/xarm_lift_medium_replay",
    "lerobot/aloha_sim_insertion_human",
    "lerobot/umi_cup_in_the_wild",
]

# Task descriptions for each dataset (used by ORBIT's capability scorer)
dataset_tasks = {
    "lerobot/pusht": ["push block to target", "precise positioning"],
    "lerobot/aloha_sim_transfer_cube_human": ["pick up cube", "bimanual handover", "place cube"],
    "lerobot/xarm_lift_medium_replay": ["grasp object", "lift object", "stable hold"],
    "lerobot/aloha_sim_insertion_human": ["align peg", "insert peg", "precision manipulation"],
    "lerobot/umi_cup_in_the_wild": ["grasp cup", "pour liquid", "place cup"],
}

# Download and inspect each dataset
dataset_info = {}
for repo_id in datasets_to_test:
    print(f"\n{'='*60}")
    print(f"Loading: {repo_id}")
    print(f"{'='*60}")
    try:
        ds = LeRobotDataset(repo_id)
        ep_index = ds.episode_data_index
        num_episodes = len(ep_index["from"])
        num_frames = len(ds)

        # Inspect a sample to get shapes
        sample = ds[0]
        obs_keys = [k for k in sample.keys() if k.startswith("observation")]
        action_dim = sample["action"].shape[0] if "action" in sample else None

        info = {
            "num_episodes": num_episodes,
            "num_frames": num_frames,
            "obs_keys": obs_keys,
            "action_dim": action_dim,
        }
        dataset_info[repo_id] = info

        print(f"  Episodes:    {num_episodes}")
        print(f"  Frames:      {num_frames}")
        print(f"  Obs keys:    {obs_keys}")
        print(f"  Action dim:  {action_dim}")

        del ds  # free memory
    except Exception as e:
        print(f"  FAILED: {e}")
        dataset_info[repo_id] = {"error": str(e)}

print(f"\n\nSuccessfully loaded {sum(1 for v in dataset_info.values() if 'error' not in v)}/{len(datasets_to_test)} datasets.")

## 3. Run ORBIT Profiler on Each Dataset

For each dataset we:
1. Convert from LeRobot format to ORBIT's HDF5 format using `DatasetLoader`
2. Run the full capability profiler (embeddings, coverage, quality, task scoring)
3. Track runtime and GPU memory usage

We limit to **20 episodes** per dataset and subsample frames (`fps_sample=2`) to stay within Colab's memory limits.

In [ ]:
from orbit.profile.profiler import DatasetProfiler
from orbit.profile.loaders import DatasetLoader

MAX_EPISODES = 20
FPS_SAMPLE = 2

profiler = DatasetProfiler(
    embedding_model="google/siglip-base-patch16-224",
    device=device,
)

profiles = {}      # repo_id -> DatasetProfile
timings = {}       # repo_id -> seconds
data_dirs = {}     # repo_id -> converted data path

for repo_id in datasets_to_test:
    if repo_id in dataset_info and "error" in dataset_info[repo_id]:
        print(f"\nSkipping {repo_id} (download failed)")
        continue

    print(f"\n{'='*60}")
    print(f"Profiling: {repo_id}")
    print(f"{'='*60}")

    output_dir = f"orbit_data/{repo_id.replace('/', '_')}" 
    os.makedirs(output_dir, exist_ok=True)

    try:
        # Step 1: Convert LeRobot -> ORBIT HDF5
        t0 = time.time()
        print(f"  Converting to ORBIT format (max {MAX_EPISODES} episodes)...")
        DatasetLoader.from_lerobot(
            repo_id, output_dir,
            max_episodes=MAX_EPISODES,
            fps_sample=FPS_SAMPLE,
        )
        data_dirs[repo_id] = output_dir

        # Step 2: Run profiler
        tasks = dataset_tasks.get(repo_id, ["general manipulation"])
        print(f"  Running profiler with tasks: {tasks}")
        profile = profiler.profile(data_dir=output_dir, task_descriptions=tasks)

        elapsed = time.time() - t0
        profiles[repo_id] = profile
        timings[repo_id] = elapsed

        # Print summary
        print(f"  Episodes profiled: {profile.num_episodes}")
        print(f"  Frames processed:  {profile.num_frames}")
        print(f"  Coverage score:    {profile.coverage.overall_coverage_score:.3f}")
        print(f"  Quality score:     {profile.quality.aggregate_score:.3f}")
        print(f"  Time:              {elapsed:.1f}s")

        if profile.capabilities:
            print(f"  Capabilities:")
            for cap in profile.capabilities:
                print(f"    - {cap.task_description}: {cap.score:.3f} (confidence: {cap.confidence:.3f})")

        # GPU memory
        if torch.cuda.is_available():
            mem_mb = torch.cuda.memory_allocated() / 1e6
            print(f"  GPU memory used:   {mem_mb:.0f} MB")
            torch.cuda.empty_cache()

    except Exception as e:
        print(f"  FAILED: {e}")
        import traceback
        traceback.print_exc()

print(f"\n\nProfiled {len(profiles)}/{len(datasets_to_test)} datasets successfully.")

## 4. Sim-to-Real Transfer Analysis

ORBIT's `Sim2RealProfiler` measures transfer readiness between two datasets by comparing:
- **Embedding overlap** — how well one dataset's visual space covers the other
- **Visual domain gap** — magnitude of visual distribution shift
- **Action similarity** — Wasserstein distance between action distributions
- **Diversity** — episode coverage breadth

Here we compare two ALOHA sim datasets (`transfer_cube` vs `insertion`) to demonstrate the analysis. In practice, you would compare a sim dataset against real-world data.

In [ ]:
from orbit.sim2real_profiler import Sim2RealProfiler

sim_repo = "lerobot/aloha_sim_transfer_cube_human"
real_repo = "lerobot/aloha_sim_insertion_human"

if sim_repo in data_dirs and real_repo in data_dirs:
    print(f"Sim-to-Real Analysis")
    print(f"  Sim dataset:  {sim_repo}")
    print(f"  Real dataset: {real_repo}")
    print(f"  (Using two sim datasets as a demonstration)")
    print()

    s2r_profiler = Sim2RealProfiler(
        embedding_model="google/siglip-base-patch16-224",
        device=device,
    )

    t0 = time.time()
    report = s2r_profiler.analyze(
        sim_dir=data_dirs[sim_repo],
        real_dir=data_dirs[real_repo],
        task_descriptions=["bimanual manipulation", "object transfer"],
    )
    s2r_time = time.time() - t0

    print(f"Overall transfer score: {report.overall_transfer_score:.3f}")
    print(f"Analysis time: {s2r_time:.1f}s")
    print()

    print("Per-task scores:")
    for task, scores in report.per_task_scores.items():
        print(f"  {task}:")
        for metric, value in scores.items():
            if isinstance(value, float):
                print(f"    {metric}: {value:.3f}")
            else:
                print(f"    {metric}: {value}")

    print()
    print("Gap analysis:")
    print(f"  Domain shift: {report.gap_analysis.get('domain_shift_summary', 'N/A')}")
    if report.gap_analysis.get("biggest_gaps"):
        print(f"  Biggest gaps:")
        for gap in report.gap_analysis["biggest_gaps"]:
            print(f"    - {gap['dimension']}: {gap['score']:.3f}")
    if report.gap_analysis.get("recommendations"):
        print(f"  Recommendations:")
        for rec in report.gap_analysis["recommendations"]:
            print(f"    - {rec}")

    print()
    print("Prescriptions:")
    for p in report.prescription:
        print(f"  #{p['priority']} [{p['task']}] — collect ~{p['estimated_demos']} demos")
        print(f"     Current: {p['current_score']:.3f} -> Target: {p['target_score']:.3f}")
        print(f"     Weakest: {p['weakest_dimension']}")

    # Save report
    with open("results/sim2real_report.json", "w") as f:
        f.write(report.to_json(indent=2))
    print("\nSaved report to results/sim2real_report.json")

    if torch.cuda.is_available():
        torch.cuda.empty_cache()
else:
    missing = [r for r in [sim_repo, real_repo] if r not in data_dirs]
    print(f"Skipping sim2real analysis — missing datasets: {missing}")

## 5. Benchmark Results

Summary table comparing all profiled datasets across key metrics.

In [ ]:
import pandas as pd

rows = []
for repo_id, profile in profiles.items():
    # Top capabilities (score >= 0.5)
    top_caps = [f"{c.task_description} ({c.score:.2f})"
                for c in sorted(profile.capabilities, key=lambda c: c.score, reverse=True)
                if c.score >= 0.5]
    # Gaps (score < 0.5)
    gaps = [f"{c.task_description} ({c.score:.2f})"
            for c in profile.capabilities if c.score < 0.5]

    rows.append({
        "Dataset": repo_id.split("/")[1],
        "Episodes": profile.num_episodes,
        "Frames": profile.num_frames,
        "Coverage": round(profile.coverage.overall_coverage_score, 3),
        "Quality": round(profile.quality.aggregate_score, 3),
        "Top Capabilities": ", ".join(top_caps) if top_caps else "None",
        "Gaps Found": ", ".join(gaps) if gaps else "None",
        "Time (s)": round(timings.get(repo_id, 0), 1),
    })

df = pd.DataFrame(rows)
display(df)

# Save to CSV
df.to_csv("results/benchmark_results.csv", index=False)
print("\nSaved to results/benchmark_results.csv")

## 6. Visualizations

### 6a. Capability Radar Charts
One radar chart per dataset showing scores across all profiled tasks.

### 6b. Cross-Dataset Heatmap
Heatmap comparing coverage, quality, and capability scores across all datasets.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams.update({"font.size": 10})

# ── 6a. Radar Charts ──────────────────────────────────────────────────

datasets_with_caps = {k: v for k, v in profiles.items() if v.capabilities}
n_charts = len(datasets_with_caps)

if n_charts > 0:
    cols = min(n_charts, 3)
    rows_count = (n_charts + cols - 1) // cols
    fig, axes = plt.subplots(rows_count, cols, figsize=(6 * cols, 5 * rows_count),
                             subplot_kw={"projection": "polar"})
    if n_charts == 1:
        axes = [axes]
    else:
        axes = list(np.array(axes).flat)

    for idx, (repo_id, profile) in enumerate(datasets_with_caps.items()):
        ax = axes[idx]
        labels = [c.task_description for c in profile.capabilities]
        scores = [c.score for c in profile.capabilities]

        # Close the polygon
        angles = np.linspace(0, 2 * np.pi, len(labels), endpoint=False).tolist()
        scores_plot = scores + [scores[0]]
        angles_plot = angles + [angles[0]]

        ax.fill(angles_plot, scores_plot, alpha=0.25, color="steelblue")
        ax.plot(angles_plot, scores_plot, "o-", color="steelblue", linewidth=2)
        ax.set_xticks(angles)
        ax.set_xticklabels(labels, size=8)
        ax.set_ylim(0, 1)
        ax.set_yticks([0.25, 0.5, 0.75, 1.0])
        ax.set_yticklabels(["0.25", "0.50", "0.75", "1.00"], size=7)
        ax.set_title(repo_id.split("/")[1], pad=20, fontsize=11, fontweight="bold")

    # Hide unused subplots
    for idx in range(n_charts, len(axes)):
        axes[idx].set_visible(False)

    plt.suptitle("ORBIT Capability Profiles per LeRobot Dataset", fontsize=14, fontweight="bold", y=1.02)
    plt.tight_layout()
    plt.savefig("results/capability_radar_charts.png", dpi=150, bbox_inches="tight")
    plt.show()
    print("Saved results/capability_radar_charts.png")
else:
    print("No capability data to plot.")

# ── 6b. Cross-Dataset Heatmap ────────────────────────────────────────

if profiles:
    # Build matrix: rows=datasets, columns=metrics
    heatmap_data = {}
    for repo_id, profile in profiles.items():
        name = repo_id.split("/")[1]
        row = {
            "Coverage": profile.coverage.overall_coverage_score,
            "Quality": profile.quality.aggregate_score,
        }
        # Add per-task capability scores
        for cap in profile.capabilities:
            row[cap.task_description] = cap.score
        heatmap_data[name] = row

    heatmap_df = pd.DataFrame(heatmap_data).T.fillna(0)

    fig, ax = plt.subplots(figsize=(max(10, len(heatmap_df.columns) * 1.2), len(heatmap_df) * 0.8 + 2))
    im = ax.imshow(heatmap_df.values, cmap="RdYlGn", aspect="auto", vmin=0, vmax=1)

    ax.set_xticks(range(len(heatmap_df.columns)))
    ax.set_xticklabels(heatmap_df.columns, rotation=45, ha="right", fontsize=9)
    ax.set_yticks(range(len(heatmap_df.index)))
    ax.set_yticklabels(heatmap_df.index, fontsize=10)

    # Annotate cells with values
    for i in range(len(heatmap_df.index)):
        for j in range(len(heatmap_df.columns)):
            val = heatmap_df.values[i, j]
            color = "white" if val < 0.4 or val > 0.8 else "black"
            ax.text(j, i, f"{val:.2f}", ha="center", va="center", color=color, fontsize=9)

    plt.colorbar(im, ax=ax, label="Score", shrink=0.8)
    ax.set_title("ORBIT Profiler Scores Across LeRobot Datasets", fontsize=13, fontweight="bold", pad=15)
    plt.tight_layout()
    plt.savefig("results/cross_dataset_heatmap.png", dpi=150, bbox_inches="tight")
    plt.show()
    print("Saved results/cross_dataset_heatmap.png")
else:
    print("No profiles to visualize.")

## 7. Export Results

Save detailed JSON reports per dataset and a combined summary suitable for README inclusion.

In [ ]:
from orbit.profile.report import ProfileReporter

reporter = ProfileReporter()

# Save per-dataset reports
all_summaries = []
for repo_id, profile in profiles.items():
    name = repo_id.split("/")[1]
    report_dict = reporter.generate_report(profile, format="dict")

    # Save individual JSON report
    report_path = f"results/{name}_report.json"
    with open(report_path, "w") as f:
        json.dump(report_dict, f, indent=2, default=str)
    print(f"Saved {report_path}")

    # Build summary entry
    all_summaries.append({
        "dataset": repo_id,
        "num_episodes": profile.num_episodes,
        "num_frames": profile.num_frames,
        "coverage_score": round(profile.coverage.overall_coverage_score, 3),
        "quality_score": round(profile.quality.aggregate_score, 3),
        "capabilities": [
            {"task": c.task_description, "score": round(c.score, 3)}
            for c in profile.capabilities
        ],
        "num_prescriptions": len(profile.prescriptions),
        "profiling_time_s": round(timings.get(repo_id, 0), 1),
    })

# Save combined summary
combined = {
    "orbit_version": "1.1.0",
    "num_datasets": len(all_summaries),
    "embedding_model": "google/siglip-base-patch16-224",
    "max_episodes_per_dataset": MAX_EPISODES,
    "device": device,
    "datasets": all_summaries,
}

with open("results/combined_summary.json", "w") as f:
    json.dump(combined, f, indent=2)
print("\nSaved results/combined_summary.json")

# Print summary for README
print("\n" + "="*60)
print("COMBINED SUMMARY (paste into README)")
print("="*60)
print(json.dumps(combined, indent=2))

# List all output files
print("\n" + "="*60)
print("OUTPUT FILES")
print("="*60)
for f in sorted(os.listdir("results")):
    size = os.path.getsize(f"results/{f}")
    print(f"  results/{f}  ({size:,} bytes)")